# Import the necesseray libraires

In [1]:
from data_handling_raw import BorderDataset, prepare_datasets_for_configs
from model_raw import ObstacleDetector
from utils_raw import EarlyStopping, TrainingMetrics, evaluate_model, save_checkpoint
from training_raw import train_with_cross_validation

import os
import json
from datetime import datetime
import traceback

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerLine2D

import onnxruntime as ort
import cv2
from glob import glob
import time

import matplotlib.pyplot as plt
import pandas as pd

# Check if device is available

In [ ]:

import torch
print(torch.__version__)           
print(torch.cuda.is_available())   
print(torch.cuda.get_device_name(0))  

2.3.0+cu121
True
NVIDIA GeForce GTX 970


# Define configurations

In [ ]:
from itertools import product

# Define base configurations for regression
base_configs = [
    {"name": "tiny_deep", "conv_channels": [8, 16, 32], "fc_size": 64, "dropout_rate": 0.1, "learning_rate": 0.0002},
    {"name": "micro_uniform", "conv_channels": [16, 16, 16], "fc_size": 64, "dropout_rate": 0.1, "learning_rate": 0.0002},
    {"name": "small_pyramid", "conv_channels": [8, 16, 32], "fc_size": 128, "dropout_rate": 0.1, "learning_rate": 0.0002},
    {"name": "efficient_micro", "conv_channels": [8, 16, 16], "fc_size": 32, "dropout_rate": 0.1, "learning_rate": 0.0002},
    {"name": "efficient_nano", "conv_channels": [4, 8, 16], "fc_size": 32, "dropout_rate": 0.1, "learning_rate": 0.0002}
]

# Common parameters
common_config = {
    "grad_clip": 0.5,
    "num_epochs": 100,
    "num_rotations": 2,
    "accumulation_steps": 1,
    "patience": 10,
    "augment_brightness_contrast": True,
    "batch_size": 64,
    "downscale_factor": 2,
    "use_efficient": False,
    "debug_mode": False,
    "downscale_factor": 2,
    "use_efficient": False,
}

def get_input_size(downscale_factor):
    return 120 // downscale_factor

# Generate permutations
variants = list(product(
    [True, ],  # augment_brightness_contrast 
    [2, 4],         # downscale_factor
    [True, False]   # use_efficient
))

configs = []
for base in base_configs:
    for augment, downscale, efficient in variants:
        config = base.copy()
        config.update(common_config)
        config["augment_brightness_contrast"] = augment
        config["downscale_factor"] = downscale
        config["use_efficient"] = efficient
        configs.append(config)

# configs = base_configs

# Print the total number of configurations
print(f"Number of configs: {len(configs)}")


Number of configs: 20


# Helper functions to run the full training pipeline

In [ ]:
def analyze_cv_results(cv_results, output_dir):
    """Analyze the results of cross-validation for regression."""
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Extract metrics
    mse_scores = cv_results['val_mse_scores']
    mae_scores = cv_results['val_mae_scores']
    huber_scores = cv_results['val_huber_scores']
    fold_metrics = cv_results['fold_metrics']
    
    # Calculate statistics
    analysis = {
        'mse': {
            'mean': float(np.mean(mse_scores)),
            'std': float(np.std(mse_scores)),
            'min': float(np.min(mse_scores)),
            'max': float(np.max(mse_scores)),
            'values': [float(x) for x in mse_scores]
        },
        'mae': {
            'mean': float(np.mean(mae_scores)),
            'std': float(np.std(mae_scores)),
            'min': float(np.min(mae_scores)),
            'max': float(np.max(mae_scores)),
            'values': [float(x) for x in mae_scores]
        },
        'huber' : {
            'mean': float(np.mean(huber_scores)),
            'std': float(np.std(huber_scores)),
            'min': float(np.min(huber_scores)),
            'max': float(np.max(huber_scores)),
            'values': [float(x) for x in huber_scores]
        },
        'fold_metrics': fold_metrics
    }
    
    # Create visualizations
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.bar(range(1, len(mse_scores)+1), mse_scores)
    plt.axhline(y=np.mean(mse_scores), color='r', linestyle='--', label=f'Mean: {np.mean(mse_scores):.4f}')
    plt.title('MSE Across Folds')
    plt.xlabel('Fold')
    plt.ylabel('MSE')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.bar(range(1, len(mae_scores)+1), mae_scores)
    plt.axhline(y=np.mean(mae_scores), color='r', linestyle='--', label=f'Mean: {np.mean(mae_scores):.4f}')
    plt.title('MAE Across Folds')
    plt.xlabel('Fold')
    plt.ylabel('MAE')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.bar(range(1, len(huber_scores)+1), huber_scores)
    plt.axhline(y=np.mean(huber_scores), color='r', linestyle='--', label=f'Mean: {np.mean(huber_scores):.4f}')
    plt.title('Huber Loss Across Folds')
    plt.xlabel('Fold')
    plt.ylabel('Huber Loss')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'cv_analysis.png'))
    plt.close()
    
    # Save analysis results
    analysis_path = os.path.join(output_dir, 'cv_analysis.json')
    with open(analysis_path, 'w') as f:
        json.dump(analysis, f, indent=2)
    
    return analysis

def compare_cv_experiments(results, output_dir):
    """
    Compare results from multiple cross-validation experiments.
    
    Args:
        results: List of experiment results
        output_dir: Directory to save comparison results
    
    Returns:
        Dictionary containing comparison results
    """
    # Extract successful experiments with CV results
    successful_cv_experiments = [r for r in results if r['status'] == 'success' and 'cv_analysis' in r]
    
    if not successful_cv_experiments:
        print("No successful cross-validation experiments to compare.")
        return None
    
    # Create comparison data
    comparison = {
        'experiment_names': [],
        'mse_mean': [],
        'mse_std': [],
        'mae_mean': [],
        'mae_std': [],
        'huber_mean': [],
        'huber_std': []
    }
    
    for exp in successful_cv_experiments:
        comparison['experiment_names'].append(exp['experiment_name'])
        comparison['mse_mean'].append(exp['cv_analysis']['mse']['mean'])
        comparison['mse_std'].append(exp['cv_analysis']['mse']['std'])
        comparison['mae_mean'].append(exp['cv_analysis']['mae']['mean'])
        comparison['mae_std'].append(exp['cv_analysis']['mae']['std'])
        comparison['huber_mean'].append(exp['cv_analysis']['huber']['mean'])
        comparison['huber_std'].append(exp['cv_analysis']['huber']['std'])
    
    # Create DataFrame for easier analysis
    df = pd.DataFrame({
        'Experiment': comparison['experiment_names'],
        'MSE Mean': comparison['mse_mean'],
        'MSE Std': comparison['mse_std'],
        'MAE Mean': comparison['mae_mean'],
        'MAE Std': comparison['mae_std'],
        'Huber Mean': comparison['huber_mean'],
        'Huber Std': comparison['huber_std']
    })
    
    # Create visualizations
    plt.figure(figsize=(15, 6))
    
    # MSE comparison
    plt.subplot(1, 3, 1)
    plt.bar(comparison['experiment_names'], comparison['mse_mean'], 
            yerr=comparison['mse_std'], capsize=10)
    plt.title('MSE Comparison Across Experiments')
    plt.ylabel('MSE')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    
    # MAE comparison
    plt.subplot(1, 3, 2)
    plt.bar(comparison['experiment_names'], comparison['mae_mean'], 
            yerr=comparison['mae_std'], capsize=10)
    plt.title('MAE Comparison Across Experiments')
    plt.ylabel('MAE')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)

    # Huber loss comparison
    plt.subplot(1, 3, 3)
    plt.bar(comparison['experiment_names'], comparison['huber_mean'],
            yerr=comparison['huber_std'], capsize=10)
    plt.title('Huber Loss Comparison Across Experiments')
    plt.ylabel('Huber Loss')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)

    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'cv_comparison.png'))
    plt.close()
    
    # Save comparison results
    comparison_path = os.path.join(output_dir, 'cv_comparison.json')
    with open(comparison_path, 'w') as f:
        json.dump(comparison, f, indent=2)
    
    # Save DataFrame as CSV
    df.to_csv(os.path.join(output_dir, 'cv_comparison.csv'), index=False)
    
    # Find the best experiment (lowest MSE)
    best_exp_idx = np.argmin(comparison['mse_mean'])
    best_experiment = {
        'name': comparison['experiment_names'][best_exp_idx],
        'mse_mean': comparison['mse_mean'][best_exp_idx],
        'mse_std': comparison['mse_std'][best_exp_idx],
        'mae_mean': comparison['mae_mean'][best_exp_idx],
        'mae_std': comparison['mae_std'][best_exp_idx],
        'huber_mean': comparison['huber_mean'][best_exp_idx],
        'huber_std': comparison['huber_std'][best_exp_idx]
    }
    
    return {
        'comparison': comparison,
        'best_experiment': best_experiment,
        'dataframe': df.to_dict('records')
    }

def run_experiment_with_cv(config, experiment_name, main_output_dir, dataset=None, n_folds=5):
    """
    Run an experiment with cross-validation for regression.
    
    Args:
        config: Configuration dictionary
        experiment_name: Name of the experiment
        main_output_dir: Main output directory
        dataset_tuple: Tuple containing (images, labels, sources)
        n_folds: Number of folds for cross-validation
    
    Returns:
        Dictionary containing experiment results
    """
    try:
        print(f"\nStarting experiment with {n_folds}-fold cross-validation: {experiment_name}")
        
        # Create experiment directory
        experiment_dir = os.path.join(main_output_dir, experiment_name)
        os.makedirs(experiment_dir, exist_ok=True)
        
        # Store the configuration
        with open(os.path.join(experiment_dir, 'config.json'), 'w') as f:
            json.dump(config, f, indent=2)
        
        # Add cross-validation to config
        cv_config = config.copy()
        cv_config['n_folds'] = n_folds
        
        # Run cross-validation
        cv_results, cv_output_dir = train_with_cross_validation(cv_config, dataset, n_folds)
        
        # Analyze cross-validation results
        cv_analysis = analyze_cv_results(cv_results, experiment_dir)
        
        return {
            'status': 'success',
            'experiment_name': experiment_name,
            'output_dir': cv_output_dir,
            'cv_analysis': cv_analysis,
            'quantized': config.get('quantize', False)
        }
        
    except Exception as e:
        print(f"Error in experiment {experiment_name} with cross-validation: {str(e)}")
        print("\nFull traceback:")
        print(traceback.format_exc())

# Run the training

In [ ]:



# Modified main code to incorporate cross-validation
def main_with_cv(configs, n_folds=5, debug=False):
    # Create timestamp at the start
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    main_output_dir = f'experiments_cv/experiment_{timestamp}'
    os.makedirs(main_output_dir, exist_ok=True)

    # Prepare all unique datasets first
    h5_path = os.path.abspath("preprocessed_labels/danger_values.h5")
    dataset_configs = prepare_datasets_for_configs(configs, h5_path, debug=debug)

    # Run all experiments with cross-validation and collect results
    results = []
    for config in configs:
        experiment_name = config['name']
        print(f"\nRunning experiment with {n_folds}-fold cross-validation: {experiment_name}")
        
        # Get the appropriate dataset based on config parameters
        dataset_key = (
            config['downscale_factor'],
            config['augment_brightness_contrast']
        )
        dataset = dataset_configs[dataset_key]
        
        # Run experiment with cross-validation
        result = run_experiment_with_cv(config, experiment_name, main_output_dir, dataset, n_folds)
        
        result['experiment_name'] = experiment_name
        result['config'] = config
        results.append(result)
        
        # Save intermediate results summary
        summary = {
            'timestamp': timestamp,
            'results': results
        }
        with open(os.path.join(main_output_dir, 'experiments_summary.json'), 'w') as f:
            json.dump(summary, f, indent=2)

    # Print final summary
    print("\n" + "="*50)
    print("Cross-Validation Experiments Summary:")
    print("="*50)

    successful_experiments = [r for r in results if r['status'] == 'success']
    failed_experiments = [r for r in results if r['status'] == 'failed']

    print(f"\nTotal experiments: {len(results)}")
    print(f"Successful experiments: {len(successful_experiments)}")
    print(f"Failed experiments: {len(failed_experiments)}")

    if successful_experiments:
        print("\nSuccessful experiments results:")
        for result in successful_experiments:
            print(f"\nExperiment: {result['experiment_name']}")
            if 'cv_analysis' in result:
                print(f"Mean MSE: {result['cv_analysis']['mse']['mean']:.4f} ± {result['cv_analysis']['mse']['std']:.4f}")
                print(f"Mean MAE: {result['cv_analysis']['mae']['mean']:.4f} ± {result['cv_analysis']['mae']['std']:.4f}")
            print(f"Output directory: {result['output_dir']}")

    if failed_experiments:
        print("\nFailed experiments:")
        for result in failed_experiments:
            print(f"\nExperiment: {result['experiment_name']}")
            print(f"Error: {result['error']}")

    # Compare all successful cross-validation experiments
    if len(successful_experiments) > 1:
        print("\nComparing cross-validation results across experiments...")
        comparison = compare_cv_experiments(results, main_output_dir)
        
        if comparison:
            best_exp = comparison['best_experiment']
            print(f"\nBest experiment: {best_exp['name']}")
            print(f"Mean MSE: {best_exp['mse_mean']:.4f} ± {best_exp['mse_std']:.4f}")
            print(f"Mean MAE: {best_exp['mae_mean']:.4f} ± {best_exp['mae_std']:.4f}")

    # Save final summary
    final_summary = {
        'timestamp': timestamp,
        'total_experiments': len(results),
        'successful_experiments': len(successful_experiments),
        'failed_experiments': len(failed_experiments),
        'results': results
    }

    # Add best model info if we have a comparison
    if len(successful_experiments) > 1 and 'comparison' in locals():
        final_summary['best_experiment'] = comparison['best_experiment']

    # Save final summary
    with open(os.path.join(main_output_dir, 'final_summary.json'), 'w') as f:
        json.dump(final_summary, f, indent=2)
    
    return main_output_dir, results

# Run the main code with cross-validation
output_dir, results = main_with_cv(configs, n_folds=3, debug=False)


Preparing dataset with parameters:
- downscale_factor: 2
- training_mode: True
Label statistics:
Mean: 0.393
Std: 0.250
Training with 9583 samples

Preparing dataset with parameters:
- downscale_factor: 4
- training_mode: True
Label statistics:
Mean: 0.393
Std: 0.250
Training with 9583 samples

Running experiment with 3-fold cross-validation: tiny_deep

Starting experiment with 3-fold cross-validation: tiny_deep
Cuda available: True


CV Folds:   0%|          | 0/3 [00:00<?, ?it/s]

Fold 1 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Fold 2 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Fold 3 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

...

CV Folds:   0%|          | 0/3 [00:00<?, ?it/s]

Fold 1 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Fold 2 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Fold 3 Epochs:   0%|          | 0/100 [00:00<?, ?it/s]


Cross-validation results (3 folds):
Average MSE: 0.0489 ± 0.0010
Average MAE: 0.1649 ± 0.0017
Average Huber: 0.0122 ± 0.0002

Cross-Validation Experiments Summary:

Total experiments: 20
Successful experiments: 20
Failed experiments: 0

Successful experiments results:

Experiment: tiny_deep
Mean MSE: 0.0318 ± 0.0015
Mean MAE: 0.1289 ± 0.0028
Output directory: results_raw/cv_training_20250323_000915

Experiment: tiny_deep
Mean MSE: 0.0344 ± 0.0031
Mean MAE: 0.1364 ± 0.0075
Output directory: results_raw/cv_training_20250323_022007

Experiment: tiny_deep
Mean MSE: 0.0341 ± 0.0052
Mean MAE: 0.1332 ± 0.0110
Output directory: results_raw/cv_training_20250323_044411

Experiment: tiny_deep
Mean MSE: 0.0375 ± 0.0003
Mean MAE: 0.1428 ± 0.0011
Output directory: results_raw/cv_training_20250323_064848

Experiment: micro_uniform
Mean MSE: 0.0339 ± 0.0011
Mean MAE: 0.1341 ± 0.0019
Output directory: results_raw/cv_training_20250323_082302

Experiment: micro_uniform
Mean MSE: 0.0369 ± 0.0015
Mean MAE

# Analyse the trained models and copy promising models to a folder for conversion to C

In [22]:
import json
import os
import shutil
import numpy as np
from pathlib import Path

def load_summary(folder):
    summary_path = os.path.join(folder, 'cv_summary.json')
    if not os.path.exists(summary_path):
        return None
    with open(summary_path, 'r') as f:
        return json.load(f)

def evaluate_model(summary):
    # Thresholds for filtering
    MAX_MSE = 0.1
    MAX_MSE_STD = 0.05

    if (summary['avg_mse'] > MAX_MSE or 
        summary['std_mse'] > MAX_MSE_STD):
        print(f"Model failed evaluation: {summary['avg_mse']:.4f} ± {summary['std_mse']:.4f}")
        return False, None
    
    # Find most representative fold (closest to mean AUC)
    fold_mses = [fold['val_mse'] for fold in summary['fold_results']]
    mean_mse = summary['avg_mse']
    best_fold_idx = int(np.argmin([abs(mse - mean_mse) for mse in fold_mses]))
    
    return True, best_fold_idx + 1  # +1 because fold numbers start at 1

def process_results(raw_results_dir, output_dir):

    
    selected_models = []
    model_counter_2 = 1
    model_counter_4 = 1
    
    # Process each experiment folder
    for folder in sorted(os.listdir(raw_results_dir)):
        folder_path = os.path.join(raw_results_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        print(f"Processing folder: {folder}")
        try:
            summary = load_summary(folder_path)
        except Exception as e:
            print(f"Error loading summary for {folder}: {str(e)}")
            continue
        if not summary:
            continue
            
        is_good_model, best_fold = evaluate_model(summary)
        if not is_good_model:
            continue
            
        # Check if the model file exists
        fold_dir = os.path.join(folder_path, f'fold_{best_fold}')
        model_file = os.path.join(fold_dir, f'best_model_fold_{best_fold}.onnx')
        cv_summary_file = os.path.join(folder_path, f'cv_summary.json')

        if not (os.path.exists(model_file)) or not (os.path.exists(cv_summary_file)):
            print(f"Files missing for {folder}")
            continue        

        # open the cv summary file and check downscaling factor
        with open(cv_summary_file, 'r') as f:
            cv_summary = json.load(f)
            downscale_factor = int(cv_summary['config']['downscale_factor'])
            model_counter = model_counter_2 if downscale_factor == 2 else model_counter_4

        output_dir_model = os.path.join(output_dir, f'downscale_{downscale_factor}')
            
        if not os.path.exists(output_dir_model):
            os.makedirs(output_dir_model)

        # Copy files with new names
        new_model_name = f'model_{model_counter}.onnx'
        new_cv_summary_name = f'cv_summary_{model_counter}.json'
        
        shutil.copy2(model_file, os.path.join(output_dir_model, new_model_name))
        shutil.copy2(cv_summary_file, os.path.join(output_dir_model, new_cv_summary_name))
        
        # Store information about selected model
        selected_models.append({
            'original_folder': folder,
            'model_number': int(model_counter),
            'fold_used': int(best_fold),  
            'avg_mse': float(summary['avg_mse']),
            'std_mse': float(summary['std_mse']),
            
        })
        if downscale_factor == 2:
            model_counter_2 += 1
        else:
            model_counter_4 += 1
    
    # Save selection summary
    summary_file = os.path.join(output_dir, 'selected_models_summary.json')
    with open(summary_file, 'w') as f:
        json.dump(selected_models, f, indent=4)
    
    return selected_models

# Run the processing
raw_results_dir = 'results_raw'
output_dir = 'promising_models_3'
selected_models = process_results(raw_results_dir, output_dir)

# Print summary of selected models
print("\nSelected Models Summary:")
print("-" * 80)
for model in selected_models:
    print(f"Model {model['model_number']}:")
    print(f"  Original folder: {model['original_folder']}")
    print(f"  Fold used: {model['fold_used']}")
    print(f"  Average MSE: {model['avg_mse']:.4f}")
    print(f"  Std MSE: {model['std_mse']:.4f}")
    print("-" * 80)

Processing folder: cv_training_20250323_000915
Processing folder: cv_training_20250323_022007
Processing folder: cv_training_20250323_044411
Processing folder: cv_training_20250323_064848
Processing folder: cv_training_20250323_082302
Processing folder: cv_training_20250323_101815
Processing folder: cv_training_20250323_120743
Processing folder: cv_training_20250323_133122
Processing folder: cv_training_20250323_144528
Processing folder: cv_training_20250323_164348
Processing folder: cv_training_20250323_183539
Processing folder: cv_training_20250323_200927
Processing folder: cv_training_20250323_213230
Processing folder: cv_training_20250323_231025
Processing folder: cv_training_20250324_011421
Processing folder: cv_training_20250324_025011
Processing folder: cv_training_20250324_042308
Processing folder: cv_training_20250324_064004
Processing folder: cv_training_20250324_084433
Processing folder: cv_training_20250324_100653

Selected Models Summary:
----------------------------------

# Conversion (outside notebook)
The models can be converted to C with the model converter utility in the paparazzi/utils folder, and speed can be tested with the benchmark script in the same folder. 

# Analyse model performance with drone speed test results
Code expects logs from drone benchmark in `logs` folder

In [2]:
import numpy as np
import onnxruntime as ort
from data_handling_raw import DangerDataset
import time
from glob import glob
from onnx_opcounter import calculate_params
import onnx
import matplotlib.pyplot as plt
import json
import os
import pandas as pd
from scipy.stats import pearsonr
from matplotlib.ticker import ScalarFormatter

def load_onnx_model(onnx_path):
    """
    Load an ONNX model for inference
    
    Args:
        onnx_path: Path to the ONNX model file
        
    Returns:
        session: ONNX Runtime inference session
    """
    
    # Create ONNX Runtime session
    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    
    # Get model input name
    input_name = session.get_inputs()[0].name
    
    # Get expected input shape
    input_shape = session.get_inputs()[0].shape
    input_size = input_shape[2] if len(input_shape) == 4 else 120  # Default to 120 if not specified
    
    print(f"Loaded ONNX model from {onnx_path}")
    print(f"Input name: {input_name}")
    print(f"Input shape: {input_shape}")
    
    return session, input_name, input_size

def speed_test_onnx(session, input_name, input_size, dataset, num_samples=100, num_iterations=100):
    """
    Perform a speed test for the ONNX model
    """
    # Randomly select indices
    indices = np.random.choice(len(dataset), size=num_samples, replace=False)
    
    # Process each sample
    total_time = 0.0
    inference_times = []
    
    for idx in indices:
        # Get the image and true danger values
        image_tensor, true_values = dataset[idx]
        
        # Convert to numpy for ONNX inference
        input_data = np.expand_dims(image_tensor.numpy(), axis=0).astype(np.float32)
        
        # Measure inference time
        start_time = time.time()
        for _ in range(num_iterations):
            predictions = session.run(None, {input_name: input_data})[0][0]
        end_time = time.time()
        
        inference_time = (end_time - start_time) / num_iterations * 1000  # Convert to milliseconds
        inference_times.append(inference_time)
        total_time += inference_time
    
    avg_time = np.mean(inference_times)
    std_time = np.std(inference_times)
    print(f"Speed test completed for {num_samples} samples and {num_iterations} iterations")
    print(f"Average inference time per sample: {avg_time:.4f}±{std_time:.4f} ms")
    return avg_time, std_time

def load_model_metrics(onnx_path):
    """
    Load the metrics for a model from its associated JSON file
    """
    base_path = os.path.dirname(onnx_path)
    model_name = os.path.basename(onnx_path)
    
    # Extract the model number
    model_num = os.path.splitext(model_name)[0].split('_')[-1]
    
    # Find the corresponding JSON file
    json_path = os.path.join(base_path, f"cv_summary_{model_num}.json")
    
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            metrics = json.load(f)
        return metrics
    else:
        print(f"Warning: No metrics file found for {model_name}")
        return None

def calculate_efficiency_score(accuracy, speed, weight_accuracy=0.6, weight_speed=0.4, df=None):
    """
    Calculate an efficiency score based on accuracy and speed
    Higher is better for both normalized metrics
    
    Args:
        accuracy: Huber loss (lower is better)
        speed: Inference time in ms (lower is better)
        weight_accuracy: Weight for accuracy in the score
        weight_speed: Weight for speed in the score
        df: DataFrame containing all models' data (for normalization)
    """
    if df is not None:
        # Dynamic normalization based on the dataset range
        min_acc = df['avg_huber'].min()
        max_acc = df['avg_huber'].max()
        
        min_speed = df['drone_inference_time'].min()
        max_speed = df['drone_inference_time'].max()
        
        # Normalize to 0-1 scale (1 is best)
        normalized_accuracy = 1.0 - ((accuracy - min_acc) / (max_acc - min_acc))
        normalized_speed = 1.0 - (np.log(speed) - np.log(min_speed)) / (np.log(max_speed) - np.log(min_speed))
    else:
        # Fallback to simple inversion with balanced scaling
        normalized_accuracy = (1.0 / accuracy) / 100
        normalized_speed = 1.0 / (speed / 10)
    
    # Calculate weighted score
    score = (weight_accuracy * normalized_accuracy) + (weight_speed * normalized_speed)
    return score * 100  # Scale to 0-100 for readability

def get_drone_inf_times(log_folder):
    # open benchmark logs folder
    logs = glob(log_folder + '/*.txt')
    inf_times = {}

    for log in logs:
        log_number = int(log.split('/')[-1].split('_')[3])
        with open(log, "r") as file:
            log_inf_times = []
            for line in file:
                split_string = line.split('-')
                if len(split_string) > 1:
                    if split_string[0] == '[OBSDET] Front processing times ':
                        log_inf_times.append(float(split_string[1].split(',')[1].split(':')[1][:-3]))
            inf_times[log_number] = np.mean(np.asarray(log_inf_times))
            
    return inf_times

def create_visualizations(df):
    """
    Create visualizations for model analysis
    """
    plt.style.use('ggplot')
    
    # Create figure with subplots  
    fig = plt.figure(figsize=(14, 10))
    
    # 1. Input size vs inference times
    ax1 = fig.add_subplot(2, 3, 1)
    scatter1 = ax1.scatter(df['input_size'], df['onnx_inference_time'], 
                          s=100, alpha=0.7, c='blue', label='ONNX')
    
    if not df['drone_inference_time'].isna().any():
        scatter2 = ax1.scatter(df['input_size'], df['drone_inference_time'], 
                              s=100, alpha=0.7, c='red', label='Drone')
    
    for i, model in df.iterrows():
        ax1.annotate(model['model_name'].split('_')[-1], 
                    (model['input_size'], model['onnx_inference_time']),
                    textcoords="offset points", xytext=(0,10), ha='center')
    
    ax1.set_xlabel('Input Size', fontsize=12)
    ax1.set_ylabel('Inference Time (ms)', fontsize=12)
    ax1.set_title('Input Size vs. Inference Time', fontsize=14)
    ax1.legend()
    
    # 2. Model parameters vs inference times
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.scatter(df['params'], df['onnx_inference_time'], 
               s=100, alpha=0.7, c='blue', label='ONNX')
    
    if not df['drone_inference_time'].isna().any():
        ax2.scatter(df['params'], df['drone_inference_time'], 
                   s=100, alpha=0.7, c='red', label='Drone')
    
    ax2.set_xlabel('Model Parameters', fontsize=12)
    ax2.set_ylabel('Inference Time (ms)', fontsize=12)
    ax2.set_title('Model Parameters vs. Inference Time', fontsize=14)
    ax2.set_xscale('log')
    ax2.legend()
    
    # 3. Accuracy vs Speed (Pareto frontier)
    if not df['avg_huber'].isna().any() and not df['drone_inference_time'].isna().any():
        ax3 = fig.add_subplot(2, 3, 3)
        scatter = ax3.scatter(df['avg_huber'], df['drone_inference_time'], 
                             s=150, alpha=0.7, c=df['efficiency_score'], cmap='viridis')
        
        for i, model in df.iterrows():
            ax3.annotate(model['model_name'].split('_')[-1], 
                         (model['avg_huber'], model['drone_inference_time']),
                         textcoords="offset points", xytext=(5,5), ha='left')
        
        ax3.set_xlabel('Huber Loss (lower is better)', fontsize=12)
        ax3.set_ylabel('Drone Inference Time (ms, lower is better)', fontsize=12)
        ax3.set_title('Accuracy vs. Speed Trade-off', fontsize=14)
        cbar = plt.colorbar(scatter, ax=ax3)
        cbar.set_label('Efficiency Score (higher is better)', fontsize=10)
        
    # 4. ONNX vs Drone inference time comparison
    if not df['drone_inference_time'].isna().any():
        ax4 = fig.add_subplot(2, 3, 4)
        ax4.scatter(df['onnx_inference_time'], df['drone_inference_time'], 
                   s=100, alpha=0.7)
        
        for i, model in df.iterrows():
            ax4.annotate(model['model_name'].split('_')[-1], 
                         (model['onnx_inference_time'], model['drone_inference_time']),
                         textcoords="offset points", xytext=(5,5), ha='left')
        
        # Add regression line
        min_x = df['onnx_inference_time'].min()
        max_x = df['onnx_inference_time'].max()
        
        if not np.isnan(min_x) and not np.isnan(max_x):
            x = np.linspace(min_x, max_x, 100)
            
            # Calculate linear regression
            m, b = np.polyfit(df['onnx_inference_time'], df['drone_inference_time'], 1)
            ax4.plot(x, m*x + b, 'k--', alpha=0.8)
            
            # Calculate and display R²
            correlation, _ = pearsonr(df['onnx_inference_time'], df['drone_inference_time'])
            r_squared = correlation**2
            ax4.text(0.05, 0.95, f'R² = {r_squared:.3f}', transform=ax4.transAxes, 
                    fontsize=12, verticalalignment='top')
        
        ax4.set_xlabel('ONNX Inference Time (ms)', fontsize=12)
        ax4.set_ylabel('Drone Inference Time (ms)', fontsize=12)
        ax4.set_title('ONNX vs. Drone Inference Time', fontsize=14)
    
    # 5. Efficiency score comparison
    if 'efficiency_score' in df.columns:
        ax5 = fig.add_subplot(2, 3, 5)
        sorted_df = df.sort_values('efficiency_score', ascending=False)
        bars = ax5.bar(sorted_df['model_name'].apply(lambda x: "_".join(x.split('_')[1:])), 
                      sorted_df['efficiency_score'], alpha=0.7)
        
        # Add value labels on top of bars
        for bar in bars:
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=9)
        
        ax5.set_xlabel('Model', fontsize=12)
        ax5.set_ylabel('Efficiency Score', fontsize=12)
        ax5.set_title('Model Efficiency Comparison', fontsize=14)
        plt.setp(ax5.get_xticklabels(), rotation=45, ha='right')
    
    # 6. Multi-metric comparison (Radar chart)
    if not df['avg_huber'].isna().any() and not df['drone_inference_time'].isna().any():
        ax6 = fig.add_subplot(2, 3, 6, polar=True)
        
        # Select top 3 models by efficiency score for radar chart
        top_models = df.sort_values('efficiency_score', ascending=False).head(3)
        
        # Categories for radar chart
        categories = ['Speed\n(1/inference)', 'Accuracy\n(1/huber)', 'Parameters\n(1/log(params))']
        N = len(categories)
        
        # Compute angle for each category
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]  # Close the polygon
        
        # Initialize the plot
        ax6.set_theta_offset(np.pi / 2)
        ax6.set_theta_direction(-1)
        ax6.set_rlabel_position(0)
        
        # Add category labels
        plt.xticks(angles[:-1], categories, fontsize=10)
        
        # Plot each model
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        for i, (_, model) in enumerate(top_models.iterrows()):
            # Normalize values between 0 and 1 for comparison
            speed_norm = 1.0 / (model['drone_inference_time'] / df['drone_inference_time'].min())
            accuracy_norm = 1.0 / (model['avg_huber'] / df['avg_huber'].min())
            params_norm = 1.0 / (np.log10(model['params']) / np.log10(df['params'].min()))
            
            values = [speed_norm, accuracy_norm, params_norm]
            values += values[:1]  # Close the polygon
            
            # Plot values
            ax6.plot(angles, values, linewidth=2, linestyle='solid', 
                    label=model['model_name'].split('_')[1], color=colors[i])
            ax6.fill(angles, values, color=colors[i], alpha=0.1)
        
        # Add legend
        ax6.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
        ax6.set_title('Top 3 Models Comparison', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    h5_path = "preprocessed_labels\\danger_values.h5"
    onnx_paths = glob("promising_models_3\\*\\*.onnx")
    print(f"Found {len(onnx_paths)} ONNX models")
    
    onnx_paths.sort()
    model_data = []
    
    drone_inf_times_downscale_2 = get_drone_inf_times('logs\\downscale_2')
    drone_inf_times_downscale_4 = get_drone_inf_times('logs\\downscale_4')
    
    # First pass: collect all model data without efficiency scores
    for _, onnx_path in enumerate(onnx_paths):
        model_name = os.path.basename(onnx_path).split('.')[0]
        print(f"\nTesting model: {model_name}")
        model_number = int(model_name.split('_')[-1])
        
        session, input_name, input_size = load_onnx_model(onnx_path)
        model_params = calculate_params(onnx.load_model(onnx_path))
        
        downscale_factor = 240 // input_size
        dataset = DangerDataset(h5_path, downscale_factor=downscale_factor, training=False)
        print(f"Loaded dataset with {len(dataset)} samples")

        onnx_inf_time, onnx_std_time = speed_test_onnx(session, input_name, input_size, dataset, num_samples=100, num_iterations=100)
        metrics = load_model_metrics(onnx_path)
        
        if downscale_factor == 2:
            drone_time = drone_inf_times_downscale_2.get(model_number, None)
        elif downscale_factor == 4:
            drone_time = drone_inf_times_downscale_4.get(model_number, None)
        
        model_info = {
            'model_name': f"{model_name}_ds_{downscale_factor}",
            'input_size': input_size,
            'params': model_params,
            'onnx_inference_time': onnx_inf_time,
            'onnx_std_time': onnx_std_time,
            'drone_inference_time': drone_time,
            'avg_huber': metrics['avg_huber'] if metrics else None,
            'avg_mae': metrics['avg_mae'] if metrics else None,
            'avg_mse': metrics['avg_mse'] if metrics else None,
        }
        
        model_data.append(model_info)
    
    # Create the DataFrame without efficiency scores
    df = pd.DataFrame(model_data)
    
    # Second pass: calculate and add efficiency scores
    if not df['avg_huber'].isna().any() and not df['drone_inference_time'].isna().any():
        # Calculate efficiency scores using the complete DataFrame
        for i, row in df.iterrows():
            df.at[i, 'efficiency_score'] = calculate_efficiency_score(
                row['avg_huber'], 
                row['drone_inference_time'],
                df=df
            )
    
    # Continue with the rest of your code
    # Calculate correlation between ONNX inference time and drone inference time
    if not df['drone_inference_time'].isna().any():
        correlation, p_value = pearsonr(df['onnx_inference_time'], df['drone_inference_time'])
        print(f"\nCorrelation between ONNX and drone inference times: {correlation:.4f} (p-value: {p_value:.4f})")
        
        # Calculate ratio between drone and ONNX times
        df['drone_onnx_ratio'] = df['drone_inference_time'] / df['onnx_inference_time']
        avg_ratio = df['drone_onnx_ratio'].mean()
        std_ratio = df['drone_onnx_ratio'].std()
        print(f"Average drone/ONNX inference time ratio: {avg_ratio:.2f}±{std_ratio:.2f}")
    
    # Print model comparison table
    print("\nModel Comparison Table:")
    comparison_df = df[['model_name', 'input_size', 'params', 'onnx_inference_time', 
                         'drone_inference_time', 'avg_huber', 'avg_mae', 'efficiency_score']]
    comparison_df = comparison_df.sort_values('efficiency_score', ascending=False)
    print(comparison_df.to_string(index=False))
    
    # Find the best model based on efficiency score
    if 'efficiency_score' in df.columns:
        best_model = df.loc[df['efficiency_score'].idxmax()]
        print(f"\nBest model based on efficiency score: {best_model['model_name']}")
        print(f"Efficiency score: {best_model['efficiency_score']:.4f}")
        print(f"Huber loss: {best_model['avg_huber']:.4f}")
        print(f"Drone inference time: {best_model['drone_inference_time']:.4f} ms")
    
    # Create visualizations
    create_visualizations(df)

    return df

def main():
    h5_path = "preprocessed_labels\\danger_values.h5"
    onnx_paths = glob("promising_models_3\\*\\*.onnx")
    print(f"Found {len(onnx_paths)} ONNX models")
    
    onnx_paths.sort()
    model_data = []
    
    drone_inf_times_downscale_2 = get_drone_inf_times('logs\\downscale_2')
    drone_inf_times_downscale_4 = get_drone_inf_times('logs\\downscale_4')
    
    # First pass: collect all model data including configuration details
    for _, onnx_path in enumerate(onnx_paths):
        model_name = os.path.basename(onnx_path).split('.')[0]
        print(f"\nTesting model: {model_name}")
        model_number = int(model_name.split('_')[-1])
        
        session, input_name, input_size = load_onnx_model(onnx_path)
        model_params = calculate_params(onnx.load_model(onnx_path))
        
        downscale_factor = 240 // input_size
        dataset = DangerDataset(h5_path, downscale_factor=downscale_factor, training=False)
        print(f"Loaded dataset with {len(dataset)} samples")

        onnx_inf_time, onnx_std_time = speed_test_onnx(session, input_name, input_size, dataset, num_samples=1, num_iterations=1)
        metrics = load_model_metrics(onnx_path)
        
        if downscale_factor == 2:
            drone_time = drone_inf_times_downscale_2.get(model_number, None)
        elif downscale_factor == 4:
            drone_time = drone_inf_times_downscale_4.get(model_number, None)
        
        # Extract model configuration details
        model_config = metrics.get('config', {}) if metrics else {}
        
        # Create a basic model info dictionary
        model_info = {
            'model_name': model_name,
            'input_size': input_size,
            'params': model_params,
            'onnx_inference_time': onnx_inf_time,
            'onnx_std_time': onnx_std_time,
            'drone_inference_time': drone_time,
            'avg_huber': metrics['avg_huber'] if metrics else None,
            'avg_mae': metrics['avg_mae'] if metrics else None,
            'avg_mse': metrics['avg_mse'] if metrics else None,
        }
        
        # Add key architectural details from configuration
        if model_config:
            # Convert conv_channels list to string for readability
            conv_channels_str = '-'.join(map(str, model_config.get('conv_channels', [])))
            
            model_info.update({
                'arch_name': model_config.get('name', 'unknown'),
                'conv_channels': conv_channels_str,
                'fc_size': model_config.get('fc_size', 0),
                'dropout_rate': model_config.get('dropout_rate', 0),
                'use_efficient': model_config.get('use_efficient', False),
                'augmentation': model_config.get('augment_brightness_contrast', False),
                'num_rotations': model_config.get('num_rotations', 0),
                'downscale_factor': model_config.get('downscale_factor', 0),
                'learning_rate': model_config.get('learning_rate', 0),
                'weight_decay': model_config.get('weight_decay', 0),
            })
        
        model_data.append(model_info)
    
    # Create the DataFrame without efficiency scores
    df = pd.DataFrame(model_data)
    
    # Second pass: calculate and add efficiency scores
    if not df['avg_huber'].isna().any() and not df['drone_inference_time'].isna().any():
        # Calculate efficiency scores using the complete DataFrame
        for i, row in df.iterrows():
            df.at[i, 'efficiency_score'] = calculate_efficiency_score(
                row['avg_huber'], 
                row['drone_inference_time'],
                df=df
            )
    
    # Calculate correlation between ONNX inference time and drone inference time
    if not df['drone_inference_time'].isna().any():
        correlation, p_value = pearsonr(df['onnx_inference_time'], df['drone_inference_time'])
        print(f"\nCorrelation between ONNX and drone inference times: {correlation:.4f} (p-value: {p_value:.4f})")
        
        # Calculate ratio between drone and ONNX times
        df['drone_onnx_ratio'] = df['drone_inference_time'] / df['onnx_inference_time']
        avg_ratio = df['drone_onnx_ratio'].mean()
        std_ratio = df['drone_onnx_ratio'].std()
        print(f"Average drone/ONNX inference time ratio: {avg_ratio:.2f}±{std_ratio:.2f}")
    
    # Sort by efficiency score
    df_sorted = df.sort_values('efficiency_score', ascending=False)
    
    # Print condensed model comparison table with key information
    print("\nModel Comparison Table (All Models):")
    # Create a condensed dataframe with the most important metrics
    condensed_df = df_sorted[['input_size', 'fc_size',  'params',
                              'use_efficient', 'conv_channels',
                              'drone_inference_time', 'avg_huber', 
                              'efficiency_score']]
    
    # Rename columns for readability
    condensed_df = condensed_df.rename(columns={
        'input_size': 'Input Size',
        'params': 'Parameters',
        'fc_size': 'FC Size',
        'use_efficient': 'Efficient',
        'conv_channels': 'Channels',
        'drone_inference_time': 'Inference (ms)',
        'avg_huber': 'Huber Loss',
        'efficiency_score': 'Efficiency'
    })
    
    # Format numbers for better readability
    condensed_df['Parameters'] = condensed_df['Parameters'].apply(lambda x: f"{x:,}")
    condensed_df['Inference (ms)'] = condensed_df['Inference (ms)'].apply(lambda x: f"{x:.2f}")
    condensed_df['Huber Loss'] = condensed_df['Huber Loss'].apply(lambda x: f"{x:.5f}")
    condensed_df['Efficiency'] = condensed_df['Efficiency'].apply(lambda x: f"{x:.2f}")
    
    # Print table
    print(condensed_df.to_string(index=False))
    
    # Print detailed comparison for top 5 models
    print("\nDetailed Comparison (Top 5 Models):")
    top5_df = df_sorted.head(5)
    detailed_df = top5_df[[
        'model_name', 'input_size', 'params', 
        'arch_name', 'conv_channels', 'fc_size', 
        'use_efficient', 'dropout_rate', 'augmentation', 
        'num_rotations', 'drone_inference_time', 
        'avg_huber', 'avg_mae', 'efficiency_score'
    ]]
    
    # Rename columns for readability
    detailed_df = detailed_df.rename(columns={
        'model_name': 'Model',
        'input_size': 'Input',
        'params': 'Params',
        'arch_name': 'Architecture',
        'conv_channels': 'Conv Channels',
        'fc_size': 'FC Size',
        'use_efficient': 'Efficient',
        'dropout_rate': 'Dropout',
        'augmentation': 'Augment',
        'num_rotations': 'Rotations',
        'drone_inference_time': 'Inference (ms)',
        'avg_huber': 'Huber Loss',
        'avg_mae': 'MAE',
        'efficiency_score': 'Efficiency'
    })
    
    # Format numbers for readability
    detailed_df['Params'] = detailed_df['Params'].apply(lambda x: f"{x:,}")
    detailed_df['Dropout'] = detailed_df['Dropout'].apply(lambda x: f"{x:.1f}")
    detailed_df['Inference (ms)'] = detailed_df['Inference (ms)'].apply(lambda x: f"{x:.2f}")
    detailed_df['Huber Loss'] = detailed_df['Huber Loss'].apply(lambda x: f"{x:.5f}")
    detailed_df['MAE'] = detailed_df['MAE'].apply(lambda x: f"{x:.3f}")
    detailed_df['Efficiency'] = detailed_df['Efficiency'].apply(lambda x: f"{x:.2f}")
    
    # Print detailed table
    print(detailed_df.to_string(index=False))
    
    # Find the best model based on efficiency score
    if 'efficiency_score' in df.columns:
        best_model = df.loc[df['efficiency_score'].idxmax()]
        print(f"\nBest model based on efficiency score: {best_model['model_name']}")
        print(f"Efficiency score: {best_model['efficiency_score']:.2f}")
        print(f"Huber loss: {best_model['avg_huber']:.5f}")
        print(f"Drone inference time: {best_model['drone_inference_time']:.2f} ms")
        # print(f"Parameters: {best_model['params']:,}")
        print(f"Architecture: {best_model.get('arch_name', 'unknown')}")
        print(f"Convolution channels: {best_model.get('conv_channels', 'unknown')}")
        print(f"Fully connected size: {best_model.get('fc_size', 'unknown')}")
        print(f"Efficient architecture: {best_model.get('use_efficient', 'unknown')}")
        print(f"Data augmentation: {best_model.get('augmentation', 'unknown')}")
    
    # Generate additional analysis to find what architecture choices correlate with performance
    print("\nArchitecture Impact Analysis:")
    
    # Analyze impact of efficient architecture
    if 'use_efficient' in df.columns:
        efficient_models = df[df['use_efficient']==True]
        standard_models = df[df['use_efficient']==False]
        
        if not efficient_models.empty and not standard_models.empty:
            print(f"Efficient architecture models (avg):")
            # print(f"  Parameters: {efficient_models['params'].mean():,.0f}")
            print(f"  Inference time: {efficient_models['drone_inference_time'].mean():.2f} ms")
            print(f"  Huber loss: {efficient_models['avg_huber'].mean():.5f}")
            print(f"  Efficiency score: {efficient_models['efficiency_score'].mean():.2f}")
            
            print(f"Standard architecture models (avg):")
            # print(f"  Parameters: {standard_models['params'].mean():,.0f}")
            print(f"  Inference time: {standard_models['drone_inference_time'].mean():.2f} ms")
            print(f"  Huber loss: {standard_models['avg_huber'].mean():.5f}")
            print(f"  Efficiency score: {standard_models['efficiency_score'].mean():.2f}")
    
    # Analyze impact of input size
    if 'input_size' in df.columns:
        size_60_models = df[df['input_size']==60]
        size_120_models = df[df['input_size']==120]
        
        if not size_60_models.empty and not size_120_models.empty:
            print(f"\nInput size 60 models (avg):")
            # print(f"  Parameters: {size_60_models['params'].mean():,.0f}")
            print(f"  Inference time: {size_60_models['drone_inference_time'].mean():.2f} ms")
            print(f"  Huber loss: {size_60_models['avg_huber'].mean():.5f}")
            print(f"  Efficiency score: {size_60_models['efficiency_score'].mean():.2f}")
            
            print(f"Input size 120 models (avg):")
            # print(f"  Parameters: {size_120_models['params'].mean():,.0f}")
            print(f"  Inference time: {size_120_models['drone_inference_time'].mean():.2f} ms")
            print(f"  Huber loss: {size_120_models['avg_huber'].mean():.5f}")
            print(f"  Efficiency score: {size_120_models['efficiency_score'].mean():.2f}")

    #  Analayse correlation between parameter count and drone inference speed
    if 'params' in df.columns and 'drone_inference_time' in df.columns:
        correlation, p_value = pearsonr(df['params'], df['drone_inference_time'])
        print(f"\nCorrelation between parameters and drone inference time: {correlation:.4f} (p-value: {p_value:.4f})")
        # Calculate ratio between parameters and drone times
        df['params_inference_ratio'] = df['params'] / df['drone_inference_time']
        avg_ratio = df['params_inference_ratio'].mean()

    
    # Save summary tables to CSV for inclusion in reports
    condensed_df.to_csv('model_comparison_summary.csv', index=False)
    detailed_df.to_csv('top5_models_detailed.csv', index=False)
    
    return df_sorted

main()


Found 20 ONNX models

Testing model: model_1
Loaded ONNX model from promising_models_3\downscale_2\model_1.onnx
Input name: input
Input shape: [1, 3, 120, 120]
Label statistics:
Mean: 0.393
Std: 0.250
Training with 9583 samples
Loaded dataset with 9583 samples
Speed test completed for 1 samples and 1 iterations
Average inference time per sample: 0.0000±0.0000 ms

Testing model: model_10
Loaded ONNX model from promising_models_3\downscale_2\model_10.onnx
Input name: input
Input shape: [1, 3, 120, 120]
Label statistics:
Mean: 0.393
Std: 0.250
Training with 9583 samples
Loaded dataset with 9583 samples
Speed test completed for 1 samples and 1 iterations
Average inference time per sample: 1.0035±0.0000 ms

Testing model: model_2
Loaded ONNX model from promising_models_3\downscale_2\model_2.onnx
Input name: input
Input shape: [1, 3, 120, 120]
Label statistics:
Mean: 0.393
Std: 0.250
Training with 9583 samples
Loaded dataset with 9583 samples
Speed test completed for 1 samples and 1 iteratio

,model_name,input_size,params,onnx_inference_time,onnx_std_time,drone_inference_time,avg_huber,avg_mae,avg_mse,arch_name,...,fc_size,dropout_rate,use_efficient,augmentation,num_rotations,downscale_factor,learning_rate,weight_decay,efficiency_score,drone_onnx_ratio
15,model_5,60,210273,1.002550,0.0,17.978061,0.008221,0.121457,0.028324,small_pyramid,...,128,0.15,True,True,2,4,0.0002,0.00005,87.168300,17.932332
10,model_1,60,103585,0.000000,0.0,13.331620,0.009309,0.133161,0.034095,tiny_deep,...,64,0.10,True,True,2,4,0.0002,0.00005,74.484552,inf
5,model_5,120,931169,1.003742,0.0,78.691623,0.008205,0.121378,0.028684,small_pyramid,...,128,0.15,True,True,2,2,0.0002,0.00005,73.144610,78.398240
0,model_1,120,464033,0.000000,0.0,65.542895,0.008883,0.128869,0.031822,tiny_deep,...,64,0.10,True,True,2,2,0.0002,0.00005,65.218636,inf
13,model_3,60,53377,0.000000,0.0,19.063038,0.009977,0.140634,0.037303,micro_uniform,...,64,0.20,True,True,2,4,0.0002,0.00005,61.469620,inf
3,model_3,120,233601,0.000000,0.0,77.579687,0.009350,0.134083,0.033859,micro_uniform,...,64,0.20,True,True,2,2,0.0002,0.00005,56.912845,inf
16,model_6,60,207251,0.000000,0.0,68.105026,0.010011,0.141246,0.037421,small_pyramid,...,128,0.15,False,True,2,4,0.0002,0.00005,48.710392,inf
12,model_2,60,106643,0.000000,0.0,65.452174,0.010127,0.142797,0.037510,tiny_deep,...,64,0.10,False,True,2,4,0.0002,0.00005,47.425442,inf
6,model_6,120,928147,0.000000,0.0,226.719899,0.009297,0.133733,0.033517,small_pyramid,...,128,0.15,False,True,2,2,0.0002,0.00005,47.324359,inf
2,model_2,120,467091,0.999689,0.0,226.538916,0.009528,0.136373,0.034422,tiny_deep,...,64,0.10,False,True,2,2,0.0002,0.00005,44.019123,226.609369
